In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import os
import glob
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import timm 
import random

CONFIG = {
    "seq_length": 20,      
    "img_size": 224,        
    "batch_size": 16,        
    "hidden_size": 128,     
    "num_layers": 2,        
    "base_model": "efficientnet_b0",
    "epochs": 15,
    "learning_rate": 1e-4,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

print(f"Device: {CONFIG['device']}")

Device: cuda


In [ ]:
class UnifiedVideoDataset(Dataset):
    def __init__(self, video_folders, labels, transform=None, seq_length=20):
        """
        video_folders: List of paths to folders containing frame images.
        labels: List of integers (0 for Fake, 1 for Real).
        """
        self.video_folders = video_folders
        self.labels = labels
        self.transform = transform
        self.seq_length = seq_length

    def __len__(self):
        return len(self.video_folders)

    def __getitem__(self, idx):
        folder_path = self.video_folders[idx]
        label = self.labels[idx]
        
        # Get all image files in the folder (sort them to keep time order)
        image_files = sorted(
            glob.glob(os.path.join(folder_path, "*.jpg")) + 
            glob.glob(os.path.join(folder_path, "*.png"))
        )
        
        # --- HANDLING EDGE CASES (Empty folders or few frames) ---
        if len(image_files) == 0:
            # Return a black video if folder is empty (avoids crashing)
            return torch.zeros((self.seq_length, 3, CONFIG['img_size'], CONFIG['img_size'])), torch.tensor(label).float()

        # --- SAMPLING STRATEGY ---
        # If we have enough frames, pick 'seq_length' evenly spaced
        if len(image_files) >= self.seq_length:
            indices = np.linspace(0, len(image_files)-1, self.seq_length).astype(int)
        else:
            # If not enough, loop the video to fill the sequence
            indices = np.pad(np.arange(len(image_files)), (0, self.seq_length - len(image_files)), mode='wrap')
            
        frames = []
        for i in indices:
            try:
                img_path = image_files[i]
                img = Image.open(img_path).convert('RGB')
                if self.transform:
                    img = self.transform(img)
                frames.append(img)
            except:
                # If an image file is corrupt, create a black frame
                frames.append(torch.zeros((3, CONFIG['img_size'], CONFIG['img_size'])))

        return torch.stack(frames), torch.tensor(label, dtype=torch.float32)

In [10]:
from sklearn.model_selection import train_test_split

# --- 1. DEFINE ROOT PATHS ---
MY_DATA_ROOT = "/kaggle/input/deep-fake-detection-extracted-some-faces/processed_data"
FF_DATA_ROOT = "/kaggle/input/faceforencispp-extracted-frames"

all_paths = []
all_labels = []

print("🔍 Scanning Datasets...")

# --- 2. GATHER ALL DATA ---
# DFD Dataset
dfd_real = glob.glob(os.path.join(MY_DATA_ROOT, "real", "*"))
dfd_fake = glob.glob(os.path.join(MY_DATA_ROOT, "fake", "*"))
all_paths.extend(dfd_real + dfd_fake)
all_labels.extend([1]*len(dfd_real) + [0]*len(dfd_fake))

# FF++ Dataset
ff_real = glob.glob(os.path.join(FF_DATA_ROOT, "real", "*"))
all_paths.extend(ff_real)
all_labels.extend([1]*len(ff_real))

ff_fake_roots = glob.glob(os.path.join(FF_DATA_ROOT, "fake", "*"))
for root in ff_fake_roots:
    fakes = glob.glob(os.path.join(root, "*"))
    all_paths.extend(fakes)
    all_labels.extend([0]*len(fakes))

print(f"✅ Total Sequences Found: {len(all_paths)}")

# --- 3. SPLIT TRAIN (80%) / VAL (20%) ---
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels, test_size=0.2, stratify=all_labels, random_state=42
)

print(f"   -> Training: {len(train_paths)} videos")
print(f"   -> Validation: {len(val_paths)} videos")

# --- 4. HANDLE IMBALANCE (TRAINING ONLY) ---
class_counts = [train_labels.count(0), train_labels.count(1)]
print(f"   -> Train Balance: {class_counts[0]} Fake vs {class_counts[1]} Real")

# Calculate weights for sampler
weight_per_class = [1.0 / class_counts[0], 1.0 / class_counts[1]]
sample_weights = [weight_per_class[label] for label in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# --- 5. CREATE DATASETS & LOADERS ---
train_transforms = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.ColorJitter(0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = UnifiedVideoDataset(train_paths, train_labels, transform=train_transforms, seq_length=CONFIG['seq_length'])
val_ds = UnifiedVideoDataset(val_paths, val_labels, transform=val_transforms, seq_length=CONFIG['seq_length'])

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

🔍 Scanning Datasets...
✅ Total Sequences Found: 6527
   -> Training: 5221 videos
   -> Validation: 1306 videos
   -> Train Balance: 4132 Fake vs 1089 Real


In [11]:
class DeepfakeDetector(nn.Module):
    def __init__(self, config):
        super(DeepfakeDetector, self).__init__()
        
        # 1. CNN Encoder (EfficientNet)
        # We use it to turn images into feature vectors
        self.backbone = timm.create_model(config['base_model'], pretrained=True)
        self.feature_dim = self.backbone.classifier.in_features
        
        # Remove the classification head of the CNN
        self.backbone.classifier = nn.Identity() 
        
        # 2. LSTM (Temporal Analysis)
        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=config['hidden_size'],
            num_layers=config['num_layers'],
            batch_first=True,
            dropout=0.4
        )
        
        # 3. Final Classifier
        self.fc = nn.Sequential(
            nn.Linear(config['hidden_size'], 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        # x shape: (batch, seq_length, 3, 224, 224)
        b, seq, c, h, w = x.size()
        
        # Flatten batch and sequence to feed into CNN
        x = x.view(b * seq, c, h, w)
        
        # Extract features using CNN
        features = self.backbone(x) # Shape: (b*seq, feature_dim)
        
        # Reshape back to sequence for LSTM
        features = features.view(b, seq, -1)
        
        # LSTM processing
        lstm_out, _ = self.lstm(features)
        
        # We take the output of the LAST frame in the sequence
        last_frame_out = lstm_out[:, -1, :]
        
        # Final prediction
        return self.fc(last_frame_out)

# Initialize Model
model = DeepfakeDetector(CONFIG)

# Handle Dual GPUs
if torch.cuda.device_count() > 1:
    print(f"🚀 Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

model = model.to(CONFIG['device'])

🚀 Using 2 GPUs!


In [ ]:
from torch.cuda.amp import GradScaler, autocast

optimizer = optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'])
criterion = nn.BCEWithLogitsLoss()
scaler = GradScaler()

# --- EARLY STOPPING CONFIG ---
patience = 3        # Stop if no improvement for 3 epochs
counter = 0         # Counts bad epochs
best_val_acc = 0.0  # Tracks best score

print("🚀 Training Started with Early Stopping...")

for epoch in range(CONFIG['epochs']):
    # --- TRAIN LOOP ---
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    loop = tqdm(train_loader, leave=True)
    for frames, labels in loop:
        frames, labels = frames.to(CONFIG['device']), labels.to(CONFIG['device']).unsqueeze(1)
        
        optimizer.zero_grad()
        with autocast():
            outputs = model(frames)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        loop.set_description(f"Epoch {epoch+1} [Train]")
        loop.set_postfix(loss=loss.item(), acc=correct/total)
    
    avg_train_acc = correct / total
    
    # --- VALIDATION LOOP ---
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0
    
    with torch.no_grad():
        for frames, labels in val_loader:
            frames, labels = frames.to(CONFIG['device']), labels.to(CONFIG['device']).unsqueeze(1)
            with autocast():
                outputs = model(frames)
                loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            
    avg_val_acc = val_correct / val_total
    print(f"📊 Epoch {epoch+1} Results: Train Acc: {avg_train_acc:.4f} | Val Acc: {avg_val_acc:.4f} | Val Loss: {val_loss/len(val_loader):.4f}")

    # --- EARLY STOPPING CHECK ---
    if avg_val_acc > best_val_acc:
        best_val_acc = avg_val_acc
        counter = 0  # Reset counter
        
        # Save the BEST model
        save_name = "best_deepfake_model.pth"
        if torch.cuda.device_count() > 1:
             torch.save(model.module.state_dict(), save_name)
        else:
             torch.save(model.state_dict(), save_name)
        print(f"🔥 New Best Model Saved! (Val Acc: {best_val_acc:.4f})")
    
    else:
        counter += 1
        print(f"⚠️ No improvement. Patience: {counter}/{patience}")
        if counter >= patience:
            print("🛑 Early Stopping Triggered. Training stopped to prevent overfitting.")
            break

print(f"🎉 Finished. Best Validation Accuracy: {best_val_acc:.4f}")